# LeetCode 797 — All Paths From Source to Target bằng Python

Link đề: https://leetcode.com/problems/all-paths-from-source-to-target/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Graph, Backtracking, Directed Acyclic Graph

Skeleton LeetCode:

```python
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: đề cho phép trả các đường đi theo **thứ tự bất kỳ**, nên trong notebook mọi phép so sánh đều **sắp xếp** kết quả (chuẩn hóa) trước khi so với đáp án.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong **đồ thị có hướng không chu trình (DAG)** `n` đỉnh, liệt kê **mọi đường đi** từ đỉnh `0` tới đỉnh `n - 1`.
2. Nắm các công cụ Python: danh sách kề `graph[u]`, sao chép list đúng cách (`path[:]`, `path + [v]`, `list(path)`), đệ quy với `append`/`pop` (backtracking), `collections.deque`, `functools.cache`, và sắp xếp topo bằng bậc vào (Kahn).
3. Đi từ **liệt kê mọi dãy đỉnh rồi kiểm tra cạnh** → **BFS giữ nguyên cả đường đi** → **DFS quay lui (backtracking)** → **DFS có nhớ (memo) trả về danh sách đường đi từ mỗi đỉnh** → **quy hoạch động theo thứ tự topo ngược**.
4. Hiểu mấu chốt: vì là **DAG** nên không cần mảng `visited` — không thể quay lại đỉnh đã đi; số đường đi có thể lên tới `2^(n-2)` nên đầu ra mới là chi phí chính.
5. Tránh các lỗi kinh điển: thêm `path` vào kết quả mà **không sao chép** (rồi bị sửa tiếp), quên `pop()` khi quay lui, dùng mảng `visited` toàn cục làm mất đường đi, và giả định cạnh luôn đi từ nhãn nhỏ đến nhãn lớn.

## 2. Đề bài & ví dụ

Cho một đồ thị **có hướng không chu trình (DAG)** gồm `n` đỉnh được đánh số từ `0` đến `n - 1`. Hãy tìm **tất cả** các đường đi từ đỉnh `0` tới đỉnh `n - 1` và trả về chúng theo **thứ tự bất kỳ**.

Đồ thị được cho dưới dạng **danh sách kề**: `graph[i]` là danh sách các đỉnh mà từ `i` có thể đi tới (có cạnh có hướng `i -> graph[i][j]`).

**Ví dụ 1**

- Input: `graph = [[1,2],[3],[3],[]]`
- Output: `[[0,1,3],[0,2,3]]`
- Giải thích: có hai đường đi: `0 -> 1 -> 3` và `0 -> 2 -> 3`.

**Ví dụ 2**

- Input: `graph = [[4,3,1],[3,2,4],[3],[4],[]]`
- Output: `[[0,4],[0,3,4],[0,1,3,4],[0,1,2,3,4],[0,1,4]]`

**Ràng buộc:**

- `n == graph.length`
- `2 <= n <= 15`
- `0 <= graph[i][j] < n`
- `graph[i][j] != i` (không có khuyên/self-loop)
- Các phần tử của `graph[i]` đều **khác nhau**
- Đồ thị đầu vào **đảm bảo là DAG**

Lưu ý: nhãn đỉnh **không** nhất thiết là thứ tự topo (cạnh có thể đi từ nhãn lớn về nhãn nhỏ). Thứ tự các đường đi trong kết quả không quan trọng.

In [1]:
from typing import List
from collections import deque
from functools import cache
from itertools import permutations
import random

def norm(paths):
    """Chuẩn hóa: mỗi đường thành tuple, rồi sắp xếp danh sách (thứ tự trả về là tùy ý)."""
    return sorted(tuple(p) for p in paths)

examples = [
    ([[1, 2], [3], [3], []], [[0, 1, 3], [0, 2, 3]]),
    ([[4, 3, 1], [3, 2, 4], [3], [4], []], [[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]),
]
for graph, expected in examples:
    print("n =", len(graph), "| graph =", graph, "| số đường kỳ vọng =", len(expected))

n = 4 | graph = [[1, 2], [3], [3], []] | số đường kỳ vọng = 2
n = 5 | graph = [[4, 3, 1], [3, 2, 4], [3], [4], []] | số đường kỳ vọng = 5


## 3. Kiến thức nền

### 3.1. Danh sách kề

`graph[u]` liệt kê các **đỉnh kề ra** của `u`. Duyệt cạnh từ `u`: `for v in graph[u]`. Đỉnh không có cạnh ra có `graph[u] == []` (như đỉnh đích `n - 1` ở các ví dụ).

In [2]:
graph = [[1, 2], [3], [3], []]
for u, nbrs in enumerate(graph):
    print(f"đỉnh {u} đi được tới {nbrs}")
edges = [(u, v) for u in range(len(graph)) for v in graph[u]]
print("danh sách cạnh:", edges, "| số cạnh:", len(edges))

đỉnh 0 đi được tới [1, 2]
đỉnh 1 đi được tới [3]
đỉnh 2 đi được tới [3]
đỉnh 3 đi được tới []
danh sách cạnh: [(0, 1), (0, 2), (1, 3), (2, 3)] | số cạnh: 4


### 3.2. DAG và vì sao không cần `visited`

**DAG** (Directed Acyclic Graph) là đồ thị có hướng **không có chu trình**: không có đường đi nào xuất phát từ một đỉnh rồi quay lại chính nó. Hệ quả: khi đi theo cạnh, ta **không bao giờ gặp lại** một đỉnh đã nằm trên đường đi hiện tại, nên không cần đánh dấu "đã thăm" để tránh vòng lặp vô hạn.

Ngược lại, nếu đồ thị có chu trình (ví dụ `0 -> 1 -> 0`), DFS không `visited` sẽ lặp mãi. Trong bài này `visited` còn **có hại**: một đỉnh có thể nằm trên **nhiều** đường đi khác nhau và ta phải đi qua nó nhiều lần.

In [3]:
# Đồ thị có chu trình 0 -> 1 -> 0: duyệt không visited sẽ không dừng (ta giới hạn độ sâu để minh họa)
cyclic = [[1], [0]]
depth_limit = 6
path = [0]
while len(path) < depth_limit:
    path.append(cyclic[path[-1]][0])
print("đường đi (bị cắt tại độ sâu 6):", path, "-> chu trình khiến đi mãi")

đường đi (bị cắt tại độ sâu 6): [0, 1, 0, 1, 0, 1] -> chu trình khiến đi mãi


### 3.3. Sao chép list: `path[:]`, `list(path)`, `path + [v]`

List là đối tượng **có thể thay đổi (mutable)**: hai biến có thể trỏ **cùng một list**. Khi lưu `path` vào kết quả rồi sau đó tiếp tục `path.append`/`path.pop`, cái đã lưu **cũng đổi theo** vì cùng một đối tượng. Phải lưu **bản sao**: `path[:]`, `list(path)` hoặc `path.copy()`. Phép `path + [v]` luôn tạo **list mới**.

In [4]:
path = [0, 1]
saved_bad = path                # cùng một đối tượng
saved_ok = path[:]              # bản sao
path.append(3)
print("sau append: path =", path)
print("saved_bad =", saved_bad, "(đổi theo!)")
print("saved_ok  =", saved_ok, "(giữ nguyên)")
print(path is saved_bad, path is saved_ok)
print("path + [9] =", path + [9], "| path vẫn =", path)

sau append: path = [0, 1, 3]
saved_bad = [0, 1, 3] (đổi theo!)
saved_ok  = [0, 1] (giữ nguyên)
True False
path + [9] = [0, 1, 3, 9] | path vẫn = [0, 1, 3]


### 3.4. Đệ quy + quay lui (`append` rồi `pop`)

**Quay lui (backtracking)**: dùng **một** list `path` chung. Đi vào đỉnh `v`: `path.append(v)`, gọi đệ quy, quay ra: `path.pop()` để trả `path` về trạng thái cũ trước khi thử nhánh khác. Nhờ vậy ta không phải tạo list mới ở mọi bước.

In [5]:
def demo(u, path, graph):
    if not graph[u]:
        print("tới lá, path =", path)
        return
    for v in graph[u]:
        path.append(v)           # đi vào v
        demo(v, path, graph)
        path.pop()               # quay ra
        print(f"  quay lui khỏi {v}, path = {path}")

p = [0]
demo(0, p, [[1, 2], [3], [3], []])
print("cuối cùng path =", p)

tới lá, path = [0, 1, 3]
  quay lui khỏi 3, path = [0, 1]
  quay lui khỏi 1, path = [0]
tới lá, path = [0, 2, 3]
  quay lui khỏi 3, path = [0, 2]
  quay lui khỏi 2, path = [0]
cuối cùng path = [0]


### 3.5. `deque`, `functools.cache` và sắp xếp topo

- `deque`: hàng đợi hai đầu, `append` thêm cuối, `popleft` lấy đầu, đều `O(1)` — dùng cho BFS.
- `@cache` (hoặc `lru_cache`): ghi nhớ kết quả hàm theo tham số. Hợp với đệ quy có **bài toán con trùng lặp** (cùng đỉnh được hỏi nhiều lần).
- **Sắp xếp topo (Kahn)**: tính **bậc vào** (`indeg`) của mỗi đỉnh; đưa các đỉnh bậc vào `0` vào hàng đợi; lấy ra một đỉnh, "xóa" cạnh ra khỏi nó (giảm bậc vào của đỉnh kề), đỉnh nào về `0` thì đưa vào hàng đợi. Thứ tự lấy ra là một thứ tự topo: **mọi cạnh `u -> v` có `u` đứng trước `v`**.

In [6]:
def topo_order(graph):
    n = len(graph)
    indeg = [0] * n
    for u in range(n):
        for v in graph[u]:
            indeg[v] += 1
    q = deque(u for u in range(n) if indeg[u] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order

print(topo_order([[1, 2], [3], [3], []]))
print(topo_order([[4, 3, 1], [3, 2, 4], [3], [4], []]))
# đồ thị mà nhãn KHÔNG theo thứ tự topo (cạnh 2 -> 0, 0 -> 1):
print(topo_order([[1], [], [0]]))

@cache
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)
print(fib(30), fib.cache_info().hits, "lần dùng lại kết quả đã nhớ")

[0, 1, 2, 3]
[0, 1, 2, 3, 4]
[2, 0, 1]
832040 28 lần dùng lại kết quả đã nhớ


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `graph = [[1,2],[3],[3],[]]`, đích là đỉnh `3`.

- Từ `0` có hai lựa chọn: sang `1` hoặc sang `2`.
- Từ `1` chỉ đi được sang `3` (đích) → được đường `0 -> 1 -> 3`.
- Từ `2` chỉ đi được sang `3` → được đường `0 -> 2 -> 3`.

Có thể vẽ **cây các lựa chọn**: gốc `0`, mỗi nhánh là một cạnh, lá là nơi dừng. Lá nào là đỉnh đích thì đường từ gốc tới lá là một đáp án.

```
0
├─ 1
│  └─ 3  ✔ (đích)   -> [0, 1, 3]
└─ 2
   └─ 3  ✔ (đích)   -> [0, 2, 3]
```

Quy luật: **mọi đường đi từ `0` tới đích** = chọn một cạnh `0 -> v`, rồi nối với **một đường đi từ `v` tới đích**. Đó là một cấu trúc đệ quy: `paths(u) = [[u] + p for v in graph[u] for p in paths(v)]`, với `paths(đích) = [[đích]]`.

In [7]:
def paths_from(graph, u, target):
    """Cài đặt thẳng công thức đệ quy ở trên (chưa tối ưu)."""
    if u == target:
        return [[u]]
    return [[u] + p for v in graph[u] for p in paths_from(graph, v, target)]

for graph, expected in examples:
    got = paths_from(graph, 0, len(graph) - 1)
    print(got)
    assert norm(got) == norm(expected)

[[0, 1, 3], [0, 2, 3]]
[[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]


## 5. Cách 1 — Brute Force: liệt kê mọi dãy đỉnh rồi kiểm tra cạnh

### Ý tưởng

Không dùng cấu trúc đồ thị: **thử mọi dãy** `0, m1, m2, ..., mk, n-1` với `m1..mk` là một **hoán vị của một tập con** các đỉnh `1..n-2` (không lặp đỉnh), và kiểm tra từng cặp liên tiếp `a, b` có cạnh `a -> b` (tức `b in graph[a]`). Lý do không lặp đỉnh và không dùng lại `0`, `n - 1` ở giữa: đồ thị là DAG nên một đường đi **không thể** thăm lại đỉnh.

### Code

In [8]:
def paths_brute(graph):
    n = len(graph)
    middle = list(range(1, n - 1))
    result = []
    for k in range(len(middle) + 1):                  # số đỉnh ở giữa
        for mids in permutations(middle, k):          # mọi dãy k đỉnh khác nhau
            path = (0,) + mids + (n - 1,)
            if all(path[i + 1] in graph[path[i]] for i in range(len(path) - 1)):
                result.append(list(path))
    return result

for graph, expected in examples:
    got = paths_brute(graph)
    print(norm(got))
    assert norm(got) == norm(expected)

[(0, 1, 3), (0, 2, 3)]
[(0, 1, 2, 3, 4), (0, 1, 3, 4), (0, 1, 4), (0, 3, 4), (0, 4)]


### Độ phức tạp và nút thắt

- Số dãy cần thử: `sum_{k=0}^{n-2} P(n-2, k)` ≈ `e * (n - 2)!` — tăng **giai thừa**. Với `n = 15`: `13!` ≈ `6 * 10^9` — không thể.
- Time `O(n! * n)`, Space `O(n)` cho một dãy.

**Nút thắt:** ta thử cả những dãy chắc chắn sai (ngay cạnh đầu tiên không tồn tại) mà không cắt sớm. Cần đi **theo cạnh thật** và bỏ nhánh sai ngay khi không có cạnh. Hãy đếm số dãy phải thử:

In [9]:
from math import perm
for n in [4, 6, 8, 10, 12, 15]:
    total = sum(perm(n - 2, k) for k in range(n - 1))
    print(f"n = {n}: số dãy brute phải thử = {total}")

n = 4: số dãy brute phải thử = 5
n = 6: số dãy brute phải thử = 65
n = 8: số dãy brute phải thử = 1957
n = 10: số dãy brute phải thử = 109601
n = 12: số dãy brute phải thử = 9864101
n = 15: số dãy brute phải thử = 16926797486


## 6. Ý tưởng thuật toán / bước đột phá

Thay vì đoán cả dãy rồi kiểm tra, ta **xây đường đi dần theo cạnh thật**: từ `0`, mỗi bước chỉ đi theo `graph[u]`. Khi tới đích `n - 1`, ghi nhận đường đi. Nhánh nào hết cạnh mà chưa tới đích thì tự kết thúc.

Vì đồ thị là **DAG**, ta không cần `visited` (không có vòng). Số lần mở rộng chỉ phụ thuộc vào **số đường đi thật** và độ dài của chúng, không còn giai thừa.

Có hai cách hiện thực quen thuộc:

1. **BFS trên đường đi**: hàng đợi chứa **cả đường đi** (không chỉ đỉnh); lấy một đường ra, mở rộng bằng mỗi đỉnh kề (`path + [v]`).
2. **DFS quay lui**: một `path` chung; `append(v)`, đệ quy, `pop()`.

Và hai cách "nhìn đệ quy" mạnh hơn: **memo** (đỉnh `u` luôn cho cùng tập đường đi tới đích, nên tính một lần) và **topo ngược** (tính các đỉnh theo thứ tự từ đích về nguồn).

In [10]:
# Đếm số đường đi thật so với số dãy brute thử: đồ thị đầy đủ theo thứ tự nhãn (i -> j với mọi i < j)
def complete_dag(n):
    return [list(range(i + 1, n)) for i in range(n)]

for n in [4, 6, 8, 10, 15]:
    g = complete_dag(n)
    @cache
    def cnt(u, g=tuple(map(tuple, g))):
        return 1 if u == len(g) - 1 else sum(cnt(v) for v in g[u])
    print(f"n = {n}: số đường đi thật = {cnt(0)} (= 2^(n-2) = {2 ** (n - 2)})")
    cnt.cache_clear()

n = 4: số đường đi thật = 4 (= 2^(n-2) = 4)
n = 6: số đường đi thật = 16 (= 2^(n-2) = 16)
n = 8: số đường đi thật = 64 (= 2^(n-2) = 64)
n = 10: số đường đi thật = 256 (= 2^(n-2) = 256)
n = 15: số đường đi thật = 8192 (= 2^(n-2) = 8192)


## 7. Các cách tiếp theo

Phần này có thêm 4 cách (tổng 5 cách): **BFS trên đường đi**, **DFS quay lui** (lời giải nộp), **DFS memo**, **topo ngược**.

### 7.1. Cách 2 — BFS trên đường đi

**Giải thích từng đoạn.**

1. `q = deque([[0]])`: hàng đợi chứa các **đường đi dở**; ban đầu chỉ có `[0]`.
2. Lấy `path = q.popleft()`, `u = path[-1]` (đỉnh cuối).
3. Nếu `u == n - 1`: đã tới đích → `ans.append(path)`, `continue`.
4. Ngược lại, với mỗi `v` trong `graph[u]`: `q.append(path + [v])` (**list mới**, không làm hỏng `path` cũ).

In [11]:
def paths_bfs(graph):
    n = len(graph)
    q = deque([[0]])
    ans = []
    while q:
        path = q.popleft()
        u = path[-1]
        if u == n - 1:
            ans.append(path)
            continue
        for v in graph[u]:
            q.append(path + [v])
    return ans

for graph, expected in examples:
    got = paths_bfs(graph)
    print(got)
    assert norm(got) == norm(expected)

[[0, 1, 3], [0, 2, 3]]
[[0, 4], [0, 3, 4], [0, 1, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4]]


In [12]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        q = deque([[0]])
        ans = []
        while q:
            path = q.popleft()
            u = path[-1]
            if u == n - 1:
                ans.append(path)
                continue
            for v in graph[u]:
                q.append(path + [v])
        return ans

print(Solution().allPathsSourceTarget([[1, 2], [3], [3], []]))

[[0, 1, 3], [0, 2, 3]]


**Độ phức tạp:** gọi `P` là số đường đi và `L` độ dài tối đa (`<= n`). Mỗi đường dở được tạo bằng cách sao chép `O(L)`; tổng số đường dở là `O(số nút trong cây lựa chọn)` ≤ `n * 2^(n-2)`. Time `O(n * 2^n)`, Space `O(n * 2^n)` (hàng đợi + đáp án).

### 7.2. Cách 3 — DFS quay lui (lời giải nộp)

**Giải thích từng đoạn.**

1. `path = [0]` là list **dùng chung** cho mọi lời gọi đệ quy; `ans = []`.
2. `dfs(u)`: nếu `u == n - 1`: `ans.append(path[:])` (**sao chép**) rồi `return`.
3. Ngược lại, với mỗi `v` trong `graph[u]`: `path.append(v)`, `dfs(v)`, `path.pop()`.
4. Gọi `dfs(0)`, trả `ans`.

Độ sâu đệ quy tối đa `n <= 15`, rất an toàn.

In [13]:
def paths_dfs(graph):
    n = len(graph)
    ans = []
    path = [0]

    def dfs(u):
        if u == n - 1:
            ans.append(path[:])          # sao chép, vì path còn bị sửa tiếp
            return
        for v in graph[u]:
            path.append(v)
            dfs(v)
            path.pop()                   # quay lui

    dfs(0)
    return ans

for graph, expected in examples:
    got = paths_dfs(graph)
    print(got)
    assert norm(got) == norm(expected)

[[0, 1, 3], [0, 2, 3]]
[[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]


In [14]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        ans = []
        path = [0]

        def dfs(u):
            if u == n - 1:
                ans.append(path[:])
                return
            for v in graph[u]:
                path.append(v)
                dfs(v)
                path.pop()

        dfs(0)
        return ans

print(Solution().allPathsSourceTarget([[4, 3, 1], [3, 2, 4], [3], [4], []]))

[[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]


**Độ phức tạp:** mỗi đường đi tìm được tốn `O(L)` để sao chép; số lần gọi `dfs` ≤ số nút của cây lựa chọn. Time `O(n * 2^n)` (tệ nhất: `2^(n-2)` đường, mỗi đường dài `<= n`), Space `O(n)` cho ngăn xếp/`path` (chưa tính đáp án). Cách này tiết kiệm bộ nhớ hơn BFS vì chỉ giữ **một** đường đang xây.

### 7.3. Cách 4 — DFS có nhớ (memo): đường đi từ mỗi đỉnh tới đích

**Ý tưởng.** Tập các đường đi từ `u` tới đích **không phụ thuộc cách ta tới được `u`**. Vậy tính `paths(u)` một lần, lưu lại: `paths(u) = [[u] + p for v in graph[u] for p in paths(v)]`, `paths(đích) = [[đích]]`. Khi nhiều đường dẫn tới cùng một đỉnh `u`, ta dùng lại kết quả. Tiết kiệm đáng kể khi đồ thị có nhiều nhánh hội tụ.

In [15]:
def paths_memo(graph):
    n = len(graph)
    memo = {}

    def f(u):
        if u == n - 1:
            return [[n - 1]]
        if u not in memo:
            memo[u] = [[u] + p for v in graph[u] for p in f(v)]
        return memo[u]

    return [p[:] for p in f(0)]

for graph, expected in examples:
    got = paths_memo(graph)
    print(got)
    assert norm(got) == norm(expected)

[[0, 1, 3], [0, 2, 3]]
[[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]


In [16]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        memo = {}

        def f(u):
            if u == n - 1:
                return [[n - 1]]
            if u not in memo:
                memo[u] = [[u] + p for v in graph[u] for p in f(v)]
            return memo[u]

        return [p[:] for p in f(0)]

print(Solution().allPathsSourceTarget([[1, 2], [3], [3], []]))

[[0, 1, 3], [0, 2, 3]]


**Độ phức tạp:** Time `O(tổng độ dài mọi đường đi lưu trong memo)` — tệ nhất vẫn `O(n * 2^n)`, nhưng **thường tốt hơn** nhiều (mỗi đỉnh tính một lần, các tổ hợp được dùng lại); Space `O(n * 2^n)` vì lưu danh sách đường đi cho từng đỉnh (kể cả đỉnh không nằm trên đường đi nào tới đích cho danh sách rỗng, nhẹ).

### 7.4. Cách 5 — Quy hoạch động theo thứ tự topo ngược

**Ý tưởng.** Dùng sắp xếp topo (mục 3.5) để lấy thứ tự mà mọi cạnh `u -> v` có `u` đứng trước `v`. Duyệt thứ tự **ngược**: khi tới `u`, mọi đỉnh kề `v` đã được tính xong, nên `paths[u] = [[u] + p for v in graph[u] for p in paths[v]]`. Cách này **không dùng đệ quy**, và **không giả định** nhãn đỉnh theo thứ tự topo.

In [17]:
def paths_topo(graph):
    n = len(graph)
    order = topo_order(graph)
    paths = [[] for _ in range(n)]
    paths[n - 1] = [[n - 1]]
    for u in reversed(order):
        if u == n - 1:
            continue
        paths[u] = [[u] + p for v in graph[u] for p in paths[v]]
    return [p[:] for p in paths[0]]

for graph, expected in examples:
    got = paths_topo(graph)
    print(got)
    assert norm(got) == norm(expected)

# đồ thị mà nhãn KHÔNG theo thứ tự topo: 0 -> 2 -> 1 -> 3 (n = 4)
g = [[2], [3], [1], []]
print(paths_topo(g), paths_dfs(g))

[[0, 1, 3], [0, 2, 3]]
[[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]
[[0, 2, 1, 3]] [[0, 2, 1, 3]]


In [18]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        indeg = [0] * n
        for u in range(n):
            for v in graph[u]:
                indeg[v] += 1
        q = deque(u for u in range(n) if indeg[u] == 0)
        order = []
        while q:
            u = q.popleft()
            order.append(u)
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        paths = [[] for _ in range(n)]
        paths[n - 1] = [[n - 1]]
        for u in reversed(order):
            if u != n - 1:
                paths[u] = [[u] + p for v in graph[u] for p in paths[v]]
        return paths[0]

print(Solution().allPathsSourceTarget([[2], [3], [1], []]))

[[0, 2, 1, 3]]


**Độ phức tạp:** sắp xếp topo `O(n + E)`; tính `paths` tổng `O(tổng độ dài các đường)`, tệ nhất `O(n * 2^n)`; Space `O(n * 2^n)`. Phù hợp khi cũng cần **đường đi từ mọi đỉnh** tới đích, hoặc khi muốn tránh đệ quy.

## 8. TRACE từng bước

### 8.1. Trace DFS quay lui với ví dụ 1

`graph = [[1,2],[3],[3],[]]`, `n = 4`, đích `3`. `path = [0]`.

| bước | hành động | `path` | `ans` |
|---|---|---|---|
| 1 | `dfs(0)`: thử `v = 1` → `append(1)` | `[0,1]` | `[]` |
| 2 | `dfs(1)`: thử `v = 3` → `append(3)` | `[0,1,3]` | `[]` |
| 3 | `dfs(3)`: tới đích → lưu bản sao | `[0,1,3]` | `[[0,1,3]]` |
| 4 | quay về `dfs(1)`: `pop()` | `[0,1]` | |
| 5 | quay về `dfs(0)`: `pop()` | `[0]` | |
| 6 | `dfs(0)`: thử `v = 2` → `append(2)` | `[0,2]` | |
| 7 | `dfs(2)`: `append(3)`, `dfs(3)` đích → lưu | `[0,2,3]` | `[[0,1,3],[0,2,3]]` |
| 8 | quay lui hết | `[0]` | |

In [19]:
def trace_dfs(graph):
    n = len(graph)
    ans = []
    path = [0]
    print(f"graph = {graph}, đích = {n - 1}")

    def dfs(u, depth):
        pad = "  " * depth
        if u == n - 1:
            ans.append(path[:])
            print(f"{pad}đích! lưu bản sao {path}  -> ans = {ans}")
            return
        print(f"{pad}dfs({u}): path = {path}, thử các đỉnh kề {graph[u]}")
        for v in graph[u]:
            path.append(v)
            dfs(v, depth + 1)
            path.pop()
            print(f"{pad}  pop {v} -> path = {path}")

    dfs(0, 0)
    return ans

res = trace_dfs([[1, 2], [3], [3], []])
assert norm(res) == [(0, 1, 3), (0, 2, 3)]

graph = [[1, 2], [3], [3], []], đích = 3
dfs(0): path = [0], thử các đỉnh kề [1, 2]
  dfs(1): path = [0, 1], thử các đỉnh kề [3]
    đích! lưu bản sao [0, 1, 3]  -> ans = [[0, 1, 3]]
    pop 3 -> path = [0, 1]
  pop 1 -> path = [0]
  dfs(2): path = [0, 2], thử các đỉnh kề [3]
    đích! lưu bản sao [0, 2, 3]  -> ans = [[0, 1, 3], [0, 2, 3]]
    pop 3 -> path = [0, 2]
  pop 2 -> path = [0]


### 8.2. Trace BFS trên đường đi với ví dụ 2

`graph = [[4,3,1],[3,2,4],[3],[4],[]]`, đích `4`. Hàng đợi ban đầu `[[0]]`.

| lần lấy | `path` lấy ra | `u` | hành động | hàng đợi sau |
|---|---|---|---|---|
| 1 | `[0]` | 0 | mở rộng 4, 3, 1 | `[0,4] [0,3] [0,1]` |
| 2 | `[0,4]` | 4 | đích → lưu | `[0,3] [0,1]` |
| 3 | `[0,3]` | 3 | mở rộng 4 | `[0,1] [0,3,4]` |
| 4 | `[0,1]` | 1 | mở rộng 3, 2, 4 | `[0,3,4] [0,1,3] [0,1,2] [0,1,4]` |
| 5 | `[0,3,4]` | 4 | đích → lưu | `[0,1,3] [0,1,2] [0,1,4]` |
| 6 | `[0,1,3]` | 3 | mở rộng 4 | `[0,1,2] [0,1,4] [0,1,3,4]` |
| 7 | `[0,1,2]` | 2 | mở rộng 3 | `[0,1,4] [0,1,3,4] [0,1,2,3]` |
| 8 | `[0,1,4]` | 4 | đích → lưu | `[0,1,3,4] [0,1,2,3]` |
| 9 | `[0,1,3,4]` | 4 | đích → lưu | `[0,1,2,3]` |
| 10 | `[0,1,2,3]` | 3 | mở rộng 4 | `[0,1,2,3,4]` |
| 11 | `[0,1,2,3,4]` | 4 | đích → lưu | rỗng |

BFS ra các đường theo thứ tự **độ dài tăng dần**: `[0,4]`, `[0,3,4]`, `[0,1,4]`, `[0,1,3,4]`, `[0,1,2,3,4]`.

In [20]:
def trace_bfs(graph):
    n = len(graph)
    q = deque([[0]])
    ans = []
    step = 0
    while q:
        path = q.popleft()
        step += 1
        u = path[-1]
        if u == n - 1:
            ans.append(path)
            print(f"lần {step}: lấy {path}: tới đích -> lưu")
            continue
        for v in graph[u]:
            q.append(path + [v])
        print(f"lần {step}: lấy {path}: mở rộng {graph[u]} | hàng đợi = {list(q)}")
    return ans

res = trace_bfs([[4, 3, 1], [3, 2, 4], [3], [4], []])
assert norm(res) == norm(examples[1][1])
print(res)

lần 1: lấy [0]: mở rộng [4, 3, 1] | hàng đợi = [[0, 4], [0, 3], [0, 1]]
lần 2: lấy [0, 4]: tới đích -> lưu
lần 3: lấy [0, 3]: mở rộng [4] | hàng đợi = [[0, 1], [0, 3, 4]]
lần 4: lấy [0, 1]: mở rộng [3, 2, 4] | hàng đợi = [[0, 3, 4], [0, 1, 3], [0, 1, 2], [0, 1, 4]]
lần 5: lấy [0, 3, 4]: tới đích -> lưu
lần 6: lấy [0, 1, 3]: mở rộng [4] | hàng đợi = [[0, 1, 2], [0, 1, 4], [0, 1, 3, 4]]
lần 7: lấy [0, 1, 2]: mở rộng [3] | hàng đợi = [[0, 1, 4], [0, 1, 3, 4], [0, 1, 2, 3]]
lần 8: lấy [0, 1, 4]: tới đích -> lưu
lần 9: lấy [0, 1, 3, 4]: tới đích -> lưu
lần 10: lấy [0, 1, 2, 3]: mở rộng [4] | hàng đợi = [[0, 1, 2, 3, 4]]
lần 11: lấy [0, 1, 2, 3, 4]: tới đích -> lưu
[[0, 4], [0, 3, 4], [0, 1, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4]]


### 8.3. Trace DP memo: dùng lại kết quả

Với `graph = [[1,2],[3],[3],[]]`: `f(3) = [[3]]`; `f(1) = [[1,3]]`; `f(2) = [[2,3]]`; `f(0) = [[0,1,3],[0,2,3]]`. Hãy xem đỉnh nào được tính một lần và được dùng lại ở đồ thị "kim cương nhiều tầng" (nhiều đường hội tụ).

In [21]:
def trace_memo(graph):
    n = len(graph)
    memo = {}
    calls = {"computed": 0, "reused": 0}

    def f(u):
        if u == n - 1:
            return [[n - 1]]
        if u in memo:
            calls["reused"] += 1
            print(f"  f({u}) dùng lại memo ({len(memo[u])} đường)")
            return memo[u]
        calls["computed"] += 1
        memo[u] = [[u] + p for v in graph[u] for p in f(v)]
        print(f"f({u}) tính xong: {len(memo[u])} đường")
        return memo[u]

    res = f(0)
    print("tính mới:", calls["computed"], "| dùng lại:", calls["reused"])
    return res

# kim cương nhiều tầng: 0 -> {1,2} -> 3 -> {4,5} -> 6 ; đỉnh 3 và 6 được hội tụ
res = trace_memo([[1, 2], [3], [3], [4, 5], [6], [6], []])
print(norm(res))
assert len(res) == 4

f(4) tính xong: 1 đường
f(5) tính xong: 1 đường
f(3) tính xong: 2 đường
f(1) tính xong: 2 đường
  f(3) dùng lại memo (2 đường)
f(2) tính xong: 2 đường
f(0) tính xong: 4 đường
tính mới: 6 | dùng lại: 1
[(0, 1, 3, 4, 6), (0, 1, 3, 5, 6), (0, 2, 3, 4, 6), (0, 2, 3, 5, 6)]


## 9. Code nộp LeetCode

Lời giải nộp là **DFS quay lui** (Cách 3): ngắn, đúng khuôn mẫu backtracking, bộ nhớ phụ `O(n)`.

In [22]:
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        ans = []
        path = [0]

        def dfs(u):
            if u == n - 1:
                ans.append(path[:])
                return
            for v in graph[u]:
                path.append(v)
                dfs(v)
                path.pop()

        dfs(0)
        return ans


sol = Solution()
for graph, expected in examples:
    got = sol.allPathsSourceTarget(graph)
    print(graph, "->", got)
    assert norm(got) == norm(expected)       # thứ tự kết quả không quan trọng

[[1, 2], [3], [3], []] -> [[0, 1, 3], [0, 2, 3]]
[[4, 3, 1], [3, 2, 4], [3], [4], []] -> [[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]


## 10. Vì sao đúng

**Mệnh đề 1 (đủ).** Mọi đường đi `0 = v0 -> v1 -> ... -> vk = n-1` trong đồ thị đều được DFS tìm thấy: tại `dfs(v_i)` vòng `for v in graph[v_i]` có `v_{i+1}` (vì có cạnh `v_i -> v_{i+1}`), nên `path` sẽ được kéo dài thành đường đi này và khi tới `v_k = n-1` thì nó được lưu.

**Mệnh đề 2 (đúng).** Mỗi lần lưu, `path` là dãy đỉnh liên tiếp theo các cạnh thật bắt đầu từ `0` và kết thúc ở `n - 1` — đúng định nghĩa đường đi.

**Mệnh đề 3 (không trùng).** Hai nhánh khác nhau ở một đỉnh có hai đỉnh kề khác nhau (đề đảm bảo `graph[i]` không trùng), nên các đường lưu ra **khác nhau**; mỗi đường được lưu **đúng một lần**.

**Mệnh đề 4 (dừng).** Đồ thị là DAG nên không có chu trình: mỗi bước đi sang đỉnh "sau" theo thứ tự topo, độ dài `path` không quá `n`; đệ quy luôn dừng, **không cần `visited`**.

**Quay lui đúng:** sau `dfs(v)`, `path.pop()` đưa `path` về đúng trạng thái trước khi `append(v)`, nên các nhánh anh em bắt đầu từ cùng một `path`. Lưu `path[:]` (bản sao) để các thay đổi sau không ảnh hưởng đáp án đã lưu.

**Số đường đi tối đa** là `2^(n-2)` (đạt được bởi đồ thị mà `i -> j` với mọi `i < j`: mỗi đỉnh ở giữa chọn "có" hoặc "không" nằm trên đường). Ta kiểm chứng bằng code.

In [23]:
# Kiểm chứng: đồ thị đầy đủ theo thứ tự nhãn có đúng 2^(n-2) đường đi, và không đồ thị DAG nào trên n đỉnh có nhiều hơn (kiểm bằng vét cạn n <= 5)
for n in range(2, 12):
    g = complete_dag(n)
    got = paths_dfs(g)
    assert len(got) == 2 ** (n - 2) and len(set(map(tuple, got))) == len(got)
print("n = 2..11: đồ thị đầy đủ có đúng 2^(n-2) đường, đều khác nhau")

from itertools import product
best = {}
for n in range(2, 6):
    pairs = [(i, j) for i in range(n) for j in range(n) if i < j]        # DAG theo nhãn (một lớp DAG đủ để thử cận)
    mx = 0
    for mask in product([0, 1], repeat=len(pairs)):
        g = [[] for _ in range(n)]
        for (i, j), bit in zip(pairs, mask):
            if bit:
                g[i].append(j)
        mx = max(mx, len(paths_dfs(g)))
    best[n] = mx
print("số đường tối đa theo vét cạn (DAG theo nhãn):", best, "= 2^(n-2):", {n: 2 ** (n - 2) for n in best})
assert all(best[n] == 2 ** (n - 2) for n in best)

n = 2..11: đồ thị đầy đủ có đúng 2^(n-2) đường, đều khác nhau
số đường tối đa theo vét cạn (DAG theo nhãn): {2: 1, 3: 2, 4: 4, 5: 8} = 2^(n-2): {2: 1, 3: 2, 4: 4, 5: 8}


## 11. Time / Space Complexity

Gọi `P` là số đường đi (`<= 2^(n-2)`), `L` độ dài tối đa (`<= n`).

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị + kiểm cạnh | `O(n! * n)` | `O(n)` |
| 2. BFS trên đường đi | `O(n * 2^n)` | `O(n * 2^n)` |
| 3. DFS quay lui (nộp) | `O(n * 2^n)` | `O(n)` (+ đáp án) |
| 4. DFS memo | `O(n * 2^n)` (thường nhỏ hơn) | `O(n * 2^n)` |
| 5. Topo ngược | `O(n * 2^n)` | `O(n * 2^n)` |

Vì **đầu ra** đã có kích thước `Θ(P * L)`, không thuật toán nào nhanh hơn `Θ(P * L)` trong trường hợp xấu. Ta đếm số lần gọi `dfs` thực tế trên đồ thị đầy đủ để thấy tăng trưởng.

In [24]:
def dfs_calls(graph):
    n = len(graph)
    calls = 0
    path = [0]

    def dfs(u):
        nonlocal calls
        calls += 1
        if u == n - 1:
            return
        for v in graph[u]:
            path.append(v)
            dfs(v)
            path.pop()

    dfs(0)
    return calls

for n in [4, 8, 12, 15]:
    print(f"n = {n}: số lần gọi dfs = {dfs_calls(complete_dag(n))}, số đường = {2 ** (n - 2)}")

n = 4: số lần gọi dfs = 8, số đường = 4
n = 8: số lần gọi dfs = 128, số đường = 64
n = 12: số lần gọi dfs = 2048, số đường = 1024
n = 15: số lần gọi dfs = 16384, số đường = 8192


## 12. Bộ test

Gồm: ví dụ đề, biên (`n = 2`, không có đường, một đường, đích có cạnh ra, nhãn không theo thứ tự topo), phản ví dụ cho lời giải sai, đối chiếu 5 cách bằng `assert` (sau khi **sắp xếp** để bỏ qua thứ tự), vét cạn mọi DAG nhỏ, test ngẫu nhiên có seed đúng ràng buộc (`n <= 15`, DAG, cạnh khác nhau), và test ở kích thước lớn nhất (`2^13` đường).

In [25]:
def count_paths(graph):
    n = len(graph)

    @cache
    def f(u):
        return 1 if u == n - 1 else sum(f(v) for v in graph[u])

    res = f(0)
    f.cache_clear()
    return res

ALL_METHODS = {
    "brute": paths_brute,          # chỉ dùng khi n nhỏ
    "bfs": paths_bfs,
    "dfs": paths_dfs,
    "memo": paths_memo,
    "topo": paths_topo,
    "leetcode": Solution().allPathsSourceTarget,
}
FAST = [k for k in ALL_METHODS if k != "brute"]

fixed_cases = [
    ([[1, 2], [3], [3], []], [[0, 1, 3], [0, 2, 3]]),
    ([[4, 3, 1], [3, 2, 4], [3], [4], []], [[0, 4], [0, 3, 4], [0, 1, 3, 4], [0, 1, 2, 3, 4], [0, 1, 4]]),
    ([[1], []], [[0, 1]]),                                  # n = 2
    ([[], []], []),                                         # n = 2, không có cạnh: không có đường
    ([[1], [], []], []),                                    # đích (đỉnh 2) không tới được
    ([[2], [3], [1], []], [[0, 2, 1, 3]]),                  # nhãn không theo thứ tự topo
    ([[1, 2], [2], []], [[0, 1, 2], [0, 2]]),
    ([[3], [0], [1], []], [[0, 3]]),                        # cạnh đi từ nhãn lớn về nhỏ (2->1->0), không liên quan đường chính
    ([[1, 2, 3], [3], [3], []], [[0, 1, 3], [0, 2, 3], [0, 3]]),
]
for graph, expected in fixed_cases:
    for name, fn in ALL_METHODS.items():
        got = fn([row[:] for row in graph])
        assert norm(got) == norm(expected), (name, graph, got, expected)
    assert count_paths(graph) == len(expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 9 bộ x 6 cách: OK


In [26]:
# Phản ví dụ 1: quên sao chép path khi lưu
def wrong_no_copy(graph):
    n = len(graph)
    ans, path = [], [0]

    def dfs(u):
        if u == n - 1:
            ans.append(path)             # SAI: lưu chính đối tượng path
            return
        for v in graph[u]:
            path.append(v)
            dfs(v)
            path.pop()

    dfs(0)
    return ans

print("không sao chép:", wrong_no_copy([[1, 2], [3], [3], []]), "| đúng:", paths_dfs([[1, 2], [3], [3], []]))

# Phản ví dụ 2: quên pop khi quay lui
def wrong_no_pop(graph):
    n = len(graph)
    ans, path = [], [0]

    def dfs(u):
        if u == n - 1:
            ans.append(path[:])
            return
        for v in graph[u]:
            path.append(v)
            dfs(v)                       # SAI: thiếu path.pop()

    dfs(0)
    return ans

print("thiếu pop     :", wrong_no_pop([[1, 2], [3], [3], []]), "| đúng:", paths_dfs([[1, 2], [3], [3], []]))

# Phản ví dụ 3: dùng visited toàn cục (mất đường đi)
def wrong_visited(graph):
    n = len(graph)
    ans, path, seen = [], [0], {0}

    def dfs(u):
        if u == n - 1:
            ans.append(path[:])
            return
        for v in graph[u]:
            if v in seen:
                continue
            seen.add(v)                  # SAI: không bỏ ra khi quay lui
            path.append(v)
            dfs(v)
            path.pop()

    dfs(0)
    return ans

print("visited toàn cục:", wrong_visited([[1, 2], [3], [3], []]), "| đúng:", paths_dfs([[1, 2], [3], [3], []]))

không sao chép: [[0], [0]] | đúng: [[0, 1, 3], [0, 2, 3]]
thiếu pop     : [[0, 1, 3], [0, 1, 3, 2, 3]] | đúng: [[0, 1, 3], [0, 2, 3]]
visited toàn cục: [[0, 1, 3]] | đúng: [[0, 1, 3], [0, 2, 3]]


In [27]:
# Sinh DAG ngẫu nhiên đúng ràng buộc: n <= 15, không self-loop, cạnh khác nhau, đảm bảo DAG (qua một thứ tự topo ẩn)
rng = random.Random(797)

def random_dag(rng, n, p, mode):
    """mode 'plain': 0 và n-1 ở vị trí tùy ý; 'ends': 0 đứng đầu, n-1 đứng cuối thứ tự topo (nhiều đường hơn)."""
    if mode == "ends":
        mid = list(range(1, n - 1))
        rng.shuffle(mid)
        order = [0] + mid + [n - 1]
    else:
        order = list(range(n))
        rng.shuffle(order)
    graph = [[] for _ in range(n)]
    for a in range(n):
        for b in range(a + 1, n):
            if rng.random() < p:
                graph[order[a]].append(order[b])
    for row in graph:
        rng.shuffle(row)                 # thứ tự kề tùy ý
    return graph

def is_dag(graph):
    return len(topo_order(graph)) == len(graph)

for _ in range(300):
    n = rng.randint(2, 15)
    g = random_dag(rng, n, rng.choice([0.2, 0.5, 0.9]), rng.choice(["plain", "ends"]))
    assert is_dag(g) and all(i not in g[i] for i in range(n)) and all(len(set(r)) == len(r) for r in g)
print("bộ sinh DAG hợp lệ")

bộ sinh DAG hợp lệ


In [28]:
# Đối chiếu mọi cách (sau khi sắp xếp), và với brute khi n <= 8; số đường khớp với DP đếm độc lập
total = 0
for _ in range(1500):
    n = rng.randint(2, 15)
    g = random_dag(rng, n, rng.choice([0.15, 0.3, 0.6, 0.9]), rng.choice(["plain", "ends"]))
    ref = norm(paths_dfs(g))
    assert len(ref) == count_paths(g), (g,)
    for name in FAST:
        assert norm(ALL_METHODS[name]([row[:] for row in g])) == ref, (name, g)
    if n <= 8:
        assert norm(paths_brute(g)) == ref, ("brute", g)
    total += 1
print("ngẫu nhiên:", total, "đồ thị: mọi cách khớp nhau, khớp DP đếm, khớp brute khi n <= 8")

# Vét cạn: mọi DAG theo nhãn (i -> j chỉ khi i < j) trên n = 4 đỉnh (2^6 = 64 đồ thị), n = 5 (2^10 = 1024)
cnt = 0
for n in (4, 5):
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    for mask in product([0, 1], repeat=len(pairs)):
        g = [[] for _ in range(n)]
        for (i, j), bit in zip(pairs, mask):
            if bit:
                g[i].append(j)
        ref = norm(paths_brute(g))
        for name in FAST:
            assert norm(ALL_METHODS[name]([row[:] for row in g])) == ref, (name, g)
        cnt += 1
print("vét cạn", cnt, "DAG nhỏ: OK")

ngẫu nhiên: 1500 đồ thị: mọi cách khớp nhau, khớp DP đếm, khớp brute khi n <= 8
vét cạn 1088 DAG nhỏ: OK


In [29]:
# Kích thước lớn nhất: n = 15, đồ thị đầy đủ -> 2^13 = 8192 đường; và phiên bản bị xáo nhãn (đảo nhãn các đỉnh giữa)
g = complete_dag(15)
for name in FAST:
    res = ALL_METHODS[name]([row[:] for row in g])
    assert len(res) == 8192 and len(set(map(tuple, res))) == 8192, name
    assert all(p[0] == 0 and p[-1] == 14 for p in res)
    assert all(b in g[a] for p in res for a, b in zip(p, p[1:]))
print("n = 15 đồ thị đầy đủ: 8192 đường, đều hợp lệ")

# xáo nhãn các đỉnh giữa: ánh xạ đỉnh k -> perm[k] (giữ 0 và 14 cố định)
rng = random.Random(7970)
mid = list(range(1, 14))
shuffled = mid[:]
rng.shuffle(shuffled)
mapping = {0: 0, 14: 14, **dict(zip(mid, shuffled))}
g2 = [[] for _ in range(15)]
for u in range(15):
    g2[mapping[u]] = [mapping[v] for v in g[u]]
assert is_dag(g2)
ref = norm(paths_dfs(g2))
for name in ["bfs", "memo", "topo", "leetcode"]:
    assert norm(ALL_METHODS[name]([row[:] for row in g2])) == ref, name
assert len(ref) == 8192
print("n = 15 nhãn bị xáo: OK")

n = 15 đồ thị đầy đủ: 8192 đường, đều hợp lệ
n = 15 nhãn bị xáo: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lưu `path` mà không sao chép

`ans.append(path)` lưu **tham chiếu** tới list đang bị sửa tiếp; cuối cùng mọi phần tử của `ans` trỏ tới cùng một list (thường là `[0]` sau khi quay lui hết). Đã thấy ở mục 12 (`wrong_no_copy`): kết quả `[[0], [0]]`. Sửa: `ans.append(path[:])`.

### Lỗi 2 — Quên `pop()` khi quay lui

Nếu chỉ `append` mà không `pop`, `path` tích lũy đỉnh của các nhánh trước và các đường đi lưu ra **sai** (dài bất thường). Cặp `append`/`pop` luôn đi đôi quanh lời gọi đệ quy.

### Lỗi 3 — Dùng `visited` toàn cục (hoặc không bỏ đánh dấu khi quay lui)

Trong bài đường đi **tất cả**, một đỉnh có thể nằm trên **nhiều** đường khác nhau; đánh dấu "đã thăm" vĩnh viễn làm mất đường. `wrong_visited` ở mục 12 chỉ tìm được một đường thay vì hai. DAG đã tự bảo vệ khỏi vòng lặp nên **không cần** `visited`.

### Lỗi 4 — Giả định cạnh luôn đi từ nhãn nhỏ đến nhãn lớn

Một số người làm DP theo thứ tự nhãn giảm dần (`for u in range(n - 1, -1, -1)`), ngầm giả định mọi cạnh `u -> v` có `v > u`. Đề **không** đảm bảo điều này (chỉ đảm bảo DAG). Với `graph = [[2],[3],[1],[]]` (đường `0 -> 2 -> 1 -> 3`), cách đó sai. Dùng DFS/memo hoặc sắp xếp topo thật.

In [30]:
def wrong_label_order_dp(graph):
    n = len(graph)
    paths = [[] for _ in range(n)]
    paths[n - 1] = [[n - 1]]
    for u in range(n - 2, -1, -1):             # SAI: giả định v > u cho mọi cạnh u -> v
        paths[u] = [[u] + p for v in graph[u] for p in paths[v]]
    return paths[0]

g = [[2], [3], [1], []]
print("DP theo nhãn:", wrong_label_order_dp(g), "| đúng:", paths_dfs(g))

DP theo nhãn: [] | đúng: [[0, 2, 1, 3]]


### Lỗi 5 — Gộp bộ nhớ đáp án với bộ nhớ chạy

Hiểu nhầm Space: `O(n)` của DFS **không tính** đáp án. Nếu đề yêu cầu "chỉ dùng `O(n)` bộ nhớ phụ", câu trả lời vẫn phải chứa `2^(n-2)` đường đi — kích thước **đầu ra** là chi phí không tránh khỏi. Hãy phân biệt "bộ nhớ phụ" và "kích thước đầu ra" khi báo cáo độ phức tạp.

### Lỗi 6 (nhỏ) — Sai biên `n = 2`

Với `n = 2`, `graph = [[1], []]`: `0` chính là đỉnh nguồn và `1` là đích; đáp án `[[0, 1]]`. Code kiểu `range(1, n - 1)` cho dãy rỗng vẫn đúng, nhưng nếu ai đó khởi tạo `ans = [[0]]` rồi mở rộng sẽ sai khi nguồn đã là đích (không xảy ra vì `n >= 2`). Luôn test `n = 2`.

In [31]:
for g in ([[1], []], [[], []]):
    print(g, "->", Solution().allPathsSourceTarget(g))

[[1], []] -> [[0, 1]]
[[], []] -> []


## 14. Mẹo & nhận dạng pattern

- **"Liệt kê tất cả đường đi / tổ hợp / hoán vị"** → **backtracking** (DFS với `append`/`pop`) và nhớ sao chép khi lưu kết quả.
- **DAG** → không cần `visited` khi liệt kê đường đi; còn khi chỉ cần **đếm** số đường thì **memo** theo đỉnh (DP trên DAG) biến `O(2^n)` thành `O(V + E)`.
- **Có sắp xếp topo** → quy hoạch động trên DAG theo thứ tự topo (xuôi hoặc ngược) không cần đệ quy.
- **BFS giữ cả đường đi** hợp khi muốn các đường ra theo **độ dài tăng dần**, nhưng tốn bộ nhớ hơn DFS.
- Khi đề nói "**thứ tự bất kỳ**": tự kiểm thử bằng cách **sắp xếp** kết quả (`sorted(tuple(p) for p in res)`) rồi so với đáp án.
- Họ bài liên quan: *Number of Ways to Arrive at Destination* (1976), *Count Paths in DAG*, *Keys and Rooms* (841), *Find Eventual Safe States* (802), *Path Sum II* (113), *Subsets* (78), *Permutations* (46).

## 15. Biến thể

### 15.1. Chỉ **đếm** số đường đi

DP trên DAG: `cnt(u) = sum(cnt(v) for v in graph[u])`, `cnt(đích) = 1`. `O(V + E)` — không cần liệt kê.

### 15.2. Đường đi **ngắn nhất** (ít cạnh nhất)

BFS thông thường theo đỉnh, `O(V + E)`; nếu muốn tất cả đường ngắn nhất, dừng ở độ dài tối thiểu tìm được.

### 15.3. Đồ thị **có chu trình**: liệt kê các **đường đi đơn** (không lặp đỉnh)

Thêm tập `on_path` các đỉnh đang nằm trên `path`: chỉ đi vào `v` nếu `v not in on_path`; **bỏ ra khỏi tập khi quay lui**.

### 15.4. Đường đi phải **đi qua** một đỉnh `k` cho trước

Lọc kết quả hoặc đếm: `cnt(0 -> k) * cnt(k -> đích)` trên DAG.

In [32]:
def shortest_path_length(graph):
    """15.2: số cạnh ít nhất từ 0 tới n-1 (BFS), -1 nếu không có."""
    n = len(graph)
    dist = [-1] * n
    dist[0] = 0
    q = deque([0])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist[n - 1]

def simple_paths_with_cycles(graph):
    """15.3: mọi đường đơn từ 0 tới n-1 trong đồ thị CÓ THỂ có chu trình."""
    n = len(graph)
    ans, path, on_path = [], [0], {0}

    def dfs(u):
        if u == n - 1:
            ans.append(path[:])
            return
        for v in graph[u]:
            if v in on_path:
                continue
            on_path.add(v)
            path.append(v)
            dfs(v)
            path.pop()
            on_path.discard(v)           # BỎ đánh dấu khi quay lui

    dfs(0)
    return ans

def count_through(graph, k):
    """15.4: số đường đi từ 0 tới n-1 đi qua đỉnh k (DAG)."""
    n = len(graph)

    def count(src, dst):
        @cache
        def f(u):
            return 1 if u == dst else sum(f(v) for v in graph[u])
        r = f(src)
        f.cache_clear()
        return r

    return count(0, k) * count(k, n - 1)

print(shortest_path_length([[4, 3, 1], [3, 2, 4], [3], [4], []]))
print(norm(simple_paths_with_cycles([[1, 2], [0, 3], [3], []])))       # có chu trình 0 <-> 1
print(count_through([[4, 3, 1], [3, 2, 4], [3], [4], []], 3))

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 9)
    g = random_dag(rng, n, 0.5, rng.choice(["plain", "ends"]))
    ref = norm(paths_dfs(g))
    lens = [len(p) - 1 for p in ref]
    assert shortest_path_length(g) == (min(lens) if lens else -1)
    assert norm(simple_paths_with_cycles(g)) == ref                  # trên DAG, đường đơn = mọi đường
    k = rng.randrange(n)
    assert count_through(g, k) == sum(1 for p in ref if k in p)
print("các biến thể 15.1-15.4 OK")

1
[(0, 1, 3), (0, 2, 3)]
3
các biến thể 15.1-15.4 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị + kiểm cạnh | Thử mọi dãy rồi kiểm | `O(n! n)` | `O(n)` | Chỉ để hiểu đề / oracle `n` nhỏ |
| 2. BFS trên đường đi | Hàng đợi chứa cả đường | `O(n 2^n)` | `O(n 2^n)` | Muốn đường ngắn ra trước |
| 3. DFS quay lui | `append` / `pop` một `path` | `O(n 2^n)` | `O(n)` + đáp án | **Nộp bài**, chuẩn mực |
| 4. DFS memo | `paths(u)` tính một lần | `O(n 2^n)` (thường nhỏ hơn) | `O(n 2^n)` | Nhiều nhánh hội tụ, cũng cần đường từ mọi đỉnh |
| 5. Topo ngược | DP từ đích về nguồn theo thứ tự topo | `O(n 2^n)` | `O(n 2^n)` | Tránh đệ quy; nhãn bất kỳ |

In [33]:
rng = random.Random(16)
g = random_dag(rng, 15, 0.6, "ends")
answers = {name: norm(ALL_METHODS[name]([row[:] for row in g])) for name in FAST}
print({name: len(v) for name, v in answers.items()}, "(số đường)")
assert len({tuple(v) for v in answers.values()}) == 1

{'bfs': 491, 'dfs': 491, 'memo': 491, 'topo': 491, 'leetcode': 491} (số đường)


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(norm(Solution().allPathsSourceTarget([[1, 2, 3], [3], [3], []])))
a = [0, 1]; b = a; c = a[:]; a.append(2)
print(b, c)
print(count_paths([[1, 2], [3], [3], []]), count_paths([[], []]))
```

In [34]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `graph = [[1,2,3],[4],[4],[4],[]]` (đích là đỉnh `4`): liệt kê **tất cả** đường đi từ `0` tới `4`, và cho biết số đường. Nếu thêm cạnh `1 -> 2` (tức `graph[1] = [4, 2]`) thì số đường thành bao nhiêu?

In [35]:
# Bài 2: điền kết quả tính tay
hand_paths = None          # list các đường (list các list)
hand_count_with_extra_edge = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS quay lui.

In [36]:
def exercise_3_paths(graph: List[List[int]]) -> List[List[int]]:
    n = len(graph)
    ans = []
    path = [0]

    def dfs(u):
        # TODO 1: nếu u là đỉnh đích (n - 1): lưu BẢN SAO của path vào ans rồi return
        # TODO 2: với mỗi v trong graph[u]:
        #     TODO 3: path.append(v), gọi dfs(v), rồi path.pop()
        pass

    dfs(0)
    return ans


# assert norm(exercise_3_paths([[1, 2], [3], [3], []])) == [(0, 1, 3), (0, 2, 3)]
# assert norm(exercise_3_paths([[1], []])) == [(0, 1)]
# assert exercise_3_paths([[], []]) == []

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 5, hãy tự viết `exercise_4_count(graph)` **đếm** số đường đi từ `0` tới `n - 1` bằng DP có nhớ (không liệt kê). Phải chạy được với `n = 15` đồ thị đầy đủ tức thì.

In [37]:
def exercise_4_count(graph: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_count([[1, 2], [3], [3], []]) == 2
# assert exercise_4_count([[], []]) == 0
# assert exercise_4_count(complete_dag(15)) == 2 ** 13

### Bài 5 — Mini challenge: đường đi trong đồ thị **có chu trình**, tối đa `k` cạnh (khó)

Đồ thị có hướng **có thể có chu trình**. Trả về mọi đường đi từ `0` tới `n - 1` có **không quá `k` cạnh** (cho phép lặp đỉnh!). Viết `exercise_5_bounded(graph, k)`.

```python
exercise_5_bounded([[1, 2], [0, 3], [3], []], 3) -> [[0,1,3], [0,2,3]]            (không có cạnh dư: các đường đi >3 cạnh bị loại)
exercise_5_bounded([[1, 2], [0, 3], [3], []], 5) -> thêm [0,1,0,1,3], [0,1,0,2,3]
```

In [38]:
def exercise_5_bounded(graph: List[List[int]], k: int) -> List[List[int]]:
    # TODO
    pass


# assert norm(exercise_5_bounded([[1, 2], [0, 3], [3], []], 3)) == [(0, 1, 3), (0, 2, 3)]
# assert norm(exercise_5_bounded([[1, 2], [0, 3], [3], []], 5)) == [(0, 1, 0, 1, 3), (0, 1, 0, 2, 3), (0, 1, 3), (0, 2, 3)]
# assert exercise_5_bounded([[1], []], 0) == []

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `b = a` cùng đối tượng; `c = a[:]` là bản sao.
- Mức 2: sau `a.append(2)`, `b` đổi theo `a` còn `c` thì không.
- Mức 3: dòng 1 `[(0, 1, 3), (0, 2, 3), (0, 3)]`; dòng 2 `[0, 1, 2] [0, 1]`; dòng 3 `2 0`.

**Bài 2**
- Mức 1: từ `0` đi sang `1`, `2` hoặc `3`, rồi cả ba đều đi sang `4`.
- Mức 2: có `3` đường: `0-1-4`, `0-2-4`, `0-3-4`.
- Mức 3: thêm cạnh `1 -> 2` tạo thêm đường `0-1-2-4` → `4` đường.

**Bài 3**
- Mức 1: giống mục 7.2.
- Mức 2: `if u == n - 1: ans.append(path[:]); return`.
- Mức 3: `for v in graph[u]: path.append(v); dfs(v); path.pop()`.

**Bài 4**
- Mức 1: `cnt(đích) = 1`; `cnt(u) = sum(cnt(v) ...)`.
- Mức 2: dùng `@cache` (hoặc dict `memo`) theo `u`.
- Mức 3: `return cnt(0)`; nhớ `n - 1` làm điều kiện cơ sở. Với `n = 15` đầy đủ, `cnt(0) = 2^13`.

**Bài 5**
- Mức 1: vì có chu trình nên **không** dùng `visited`; thay vào đó dùng **giới hạn độ dài** để dừng.
- Mức 2: `dfs(u, steps)`: nếu `u == n - 1` thì lưu (và **vẫn có thể đi tiếp**? — với đề này, đích là nơi kết thúc đường nên `return`); nếu `steps == k` thì `return`.
- Mức 3: `for v in graph[u]: path.append(v); dfs(v, steps + 1); path.pop()`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
print(norm(Solution().allPathsSourceTarget([[1, 2, 3], [3], [3], []])))
a = [0, 1]; b = a; c = a[:]; a.append(2)
print(b, c)
print(count_paths([[1, 2], [3], [3], []]), count_paths([[], []]))

assert norm(Solution().allPathsSourceTarget([[1, 2, 3], [3], [3], []])) == [(0, 1, 3), (0, 2, 3), (0, 3)]
assert (b, c) == ([0, 1, 2], [0, 1])
assert (count_paths([[1, 2], [3], [3], []]), count_paths([[], []])) == (2, 0)

[(0, 1, 3), (0, 2, 3), (0, 3)]
[0, 1, 2] [0, 1]
2 0


### Đáp án Bài 2

In [40]:
g = [[1, 2, 3], [4], [4], [4], []]
res = norm(Solution().allPathsSourceTarget(g))
print(res)
assert res == [(0, 1, 4), (0, 2, 4), (0, 3, 4)]
g2 = [[1, 2, 3], [4, 2], [4], [4], []]
res2 = norm(Solution().allPathsSourceTarget(g2))
print(res2)
assert len(res2) == 4 and (0, 1, 2, 4) in res2

[(0, 1, 4), (0, 2, 4), (0, 3, 4)]
[(0, 1, 2, 4), (0, 1, 4), (0, 2, 4), (0, 3, 4)]


### Đáp án Bài 3

In [41]:
def exercise_3_solution(graph: List[List[int]]) -> List[List[int]]:
    n = len(graph)
    ans = []
    path = [0]

    def dfs(u):
        if u == n - 1:
            ans.append(path[:])
            return
        for v in graph[u]:
            path.append(v)
            dfs(v)
            path.pop()

    dfs(0)
    return ans

assert norm(exercise_3_solution([[1, 2], [3], [3], []])) == [(0, 1, 3), (0, 2, 3)]
assert norm(exercise_3_solution([[1], []])) == [(0, 1)]
assert exercise_3_solution([[], []]) == []
rng = random.Random(33)
for _ in range(200):
    g = random_dag(rng, rng.randint(2, 10), 0.5, "ends")
    assert norm(exercise_3_solution(g)) == norm(paths_bfs(g))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_solution(graph: List[List[int]]) -> int:
    n = len(graph)

    @cache
    def cnt(u):
        if u == n - 1:
            return 1
        return sum(cnt(v) for v in graph[u])

    result = cnt(0)
    cnt.cache_clear()
    return result

assert exercise_4_solution([[1, 2], [3], [3], []]) == 2
assert exercise_4_solution([[], []]) == 0
assert exercise_4_solution(complete_dag(15)) == 2 ** 13
rng = random.Random(44)
for _ in range(300):
    g = random_dag(rng, rng.randint(2, 12), 0.5, rng.choice(["plain", "ends"]))
    assert exercise_4_solution(g) == len(paths_dfs(g))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_solution(graph: List[List[int]], k: int) -> List[List[int]]:
    n = len(graph)
    ans = []
    path = [0]

    def dfs(u, steps):
        if u == n - 1:
            ans.append(path[:])
            return
        if steps == k:
            return
        for v in graph[u]:
            path.append(v)
            dfs(v, steps + 1)
            path.pop()

    dfs(0, 0)
    return ans

g = [[1, 2], [0, 3], [3], []]
assert norm(exercise_5_solution(g, 3)) == [(0, 1, 3), (0, 2, 3)]
assert norm(exercise_5_solution(g, 5)) == [(0, 1, 0, 1, 3), (0, 1, 0, 2, 3), (0, 1, 3), (0, 2, 3)]
assert exercise_5_solution([[1], []], 0) == []

# oracle độc lập: BFS trên đường đi giới hạn độ dài
def bounded_oracle(graph, k):
    n = len(graph)
    q = deque([[0]])
    out = []
    while q:
        p = q.popleft()
        if p[-1] == n - 1:
            out.append(p)
            continue
        if len(p) - 1 == k:
            continue
        for v in graph[p[-1]]:
            q.append(p + [v])
    return out

rng = random.Random(55)
for _ in range(300):
    n = rng.randint(2, 6)
    g = [[v for v in range(n) if v != u and rng.random() < 0.4] for u in range(n)]   # có thể có chu trình
    k = rng.randint(0, 5)
    assert norm(exercise_5_solution(g, k)) == norm(bounded_oracle(g, k))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template DFS quay lui

```python
class Solution:
    def allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]:
        n = len(graph)
        ans = []
        path = [0]                       # MỘT list dùng chung

        def dfs(u):
            if u == n - 1:
                ans.append(path[:])      # lưu BẢN SAO
                return
            for v in graph[u]:
                path.append(v)           # đi vào v
                dfs(v)
                path.pop()               # quay lui

        dfs(0)
        return ans
```

Câu thần chú: **"đi vào thì `append`, đi ra thì `pop`; tới đích thì lưu bản sao; DAG thì không cần `visited`."**

### Checklist trước khi submit

- [ ] Lưu `path[:]` (hoặc `list(path)`), không lưu `path`.
- [ ] Mỗi `append` đều có `pop` tương ứng sau lời gọi đệ quy.
- [ ] **Không** dùng `visited` toàn cục (DAG, và còn mất đường đi).
- [ ] Không giả định cạnh luôn đi từ nhãn nhỏ đến lớn.
- [ ] Test `n = 2` (có cạnh và không có cạnh), đồ thị không có đường, đồ thị đầy đủ `n = 15` (8192 đường).
- [ ] Khi tự kiểm thử, **sắp xếp** kết quả trước khi so sánh vì thứ tự không quan trọng.
- [ ] Skeleton đúng: `class Solution`, method `allPathsSourceTarget(self, graph: List[List[int]]) -> List[List[int]]`.